# 🏗️ Étape 3 — Construction des Datasets (CPT & SFT), Rétro-Instruction & Push Hugging Face 💻 *(Exécution Locale)*

**Environnement** : Ce notebook tourne **100% en local sur ton PC (CPU)**.

**Objectif pédagogique** : Assembler tes différentes sources locales (`../conv_bot.json`, `../data/processed/filtered_whatsapp_sft.json`, et les arbres YouTube/TikTok), créer des paires synthétiques par **Rétro-Instruction (*Back-translation*)**, séparer proprement un jeu de test (`sft_test`), et pousser le tout sur **Hugging Face Hub** depuis ton PC pour que le notebook Colab d'entraînement (Étape 4) puisse le télécharger en 3 secondes !

> 📖 **Conseil** : Lis `03_dataset_builder/README.md` avant de démarrer.

In [24]:
import os
import re
import json
import random
from pathlib import Path
from typing import List, Dict, Tuple
from dotenv import load_dotenv
from datasets import Dataset, DatasetDict

PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

FINAL_DATA_DIR = PROJECT_ROOT / "data" / "final"
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
random.seed(3407)
print(f"✅ Dossier de sortie local prêt : {FINAL_DATA_DIR}")

c:\Users\arist\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dossier de sortie local prêt : C:\Users\arist\Documents\Mes Projets IA\CamfranglaisLLM\CamfranglaisLLM\data\final


---
## 1. Filtre de Qualité Dialogique sur les Paires `Parent -> Enfant` (`TODO 1`)

Complète le **`TODO 1`** pour sélectionner uniquement les paires `Parent -> Enfant` où :
1. Le message `Parent` (`user`) pose une question (contient `?` ou un mot interrogatif) ou fait au moins 6 mots.
2. La réponse `Enfant` (`assistant`) fait au moins `min_assistant_words` mots (ex: 8 mots) et a un score Camfranglais $S \ge \tau_{\text{haut}}$.

In [10]:
# === EXERCICE GUIDÉ : TODO 1 ===
INTERROGATIVE_MARKERS = {"pourquoi", "comment", "qui", "quand", "où", "ou", "quel", "quelle", "quels", "quelles", "combien", "wanda"}

def is_good_sft_dialogue_pair(
    user_text: str,
    assistant_text: str,
    assistant_score: float,
    min_score: float = 0.15,
    min_assistant_words: int = 8
) -> bool:
    """
    Vérifie si une paire (user_text, assistant_text) issue des réseaux sociaux
    constitue un bon exemple d'entraînement SFT pour notre chatbot.
    """
    user_words = re.findall(r"[a-zA-ZÀ-ÿ']+", user_text.lower())
    assistant_words = re.findall(r"[a-zA-ZÀ-ÿ']+", assistant_text.lower())


    # 1. Vérifier que `assistant_score >= min_score` et `len(assistant_words) >= min_assistant_words`.
    if assistant_score < min_score and len(assistant_words) < min_assistant_words:
        return False
    
    # 2. Vérifier que `user_text` contient '?' OU qu'au moins un mot de `user_words` est dans `INTERROGATIVE_MARKERS`
    #    OU que `len(user_words) >= 6`.
    if '?'not in user_text or not any (word in INTERROGATIVE_MARKERS for word in user_words) or len(user_words) < 6:
        return False

    return True

In [16]:
# 🧪 Cellule de validation du TODO 1
assert is_good_sft_dialogue_pair(
    "Pourquoi il n'a pas sorti le son hier ?",
    "Gars le mbom a ndem, il a préféré go gérer ses ngata au quartier.",
    assistant_score=0.28
) is True

assert is_good_sft_dialogue_pair(
    "Pourquoi il n'a pas sorti le son hier ?",
    "Trop vrai mbom je valide !",
    assistant_score=0.30
) is True

print("🎉 TODO 1 validé !")

🎉 TODO 1 validé !


---
## 2. Génération Synthétique par Rétro-Instruction / *Back-Translation* (`TODO 2`)

Complète le **`TODO 2`** pour construire le prompt de rétro-instruction et formater le résultat au format `conversations`.

In [20]:
# === EXERCICE GUIDÉ : TODO 2 ===
def build_backtranslation_prompt(camfranglais_response: str) -> str:
    """
    Construit la consigne envoyée à un LLM pour générer l'instruction/question utilisateur
    qui aurait provoqué `camfranglais_response`.
    """
    # TODO 2.1 : Rédige le prompt demandant au modèle de générer UNIQUEMENT le message
    # d'un utilisateur (en français ou en camfranglais) qui aurait naturellement provoqué
    # la réponse `camfranglais_response` dans une discussion entre amis.
    
    # --- TON CODE ICI ---
    prompt = f"""Tu es un assistant qui spécialisé dans la reconstruction des questions en te basant sur les réponses données .
    Ta tâche est de générer UNIQUEMENT le message d'un utilisateur (en français ou en camfranglais) qui aurait naturellement provoqué la réponse suivante dans une discussion entre amis :
    Réponse : "{camfranglais_response}" """
    # --------------------
    return prompt


def create_sft_entry(user_message: str, assistant_response: str, source: str = "synthetic_backtranslation") -> Dict:
    """
    Formate une paire (user_message, assistant_response) au format standard :
    {
      "conversations": [
        {"role": "user", "content": ...},
        {"role": "assistant", "content": ...}
      ],
      "source": source
    }
    """
    # TODO 2.2 : Retourner le dictionnaire structuré en nettoyant les espaces (.strip())
    user_message = user_message.strip()
    assistant_response = assistant_response. strip()
    source = source.strip()
    dict_entry = {
        "conversations": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_response}
        ],
        "source": source
    }

    # --- TON CODE ICI ---
    return dict_entry
    # --------------------

In [21]:
# 🧪 Cellule de validation du TODO 2
sample_resp = "Gars, le pays est dur, même l'huile rouge est grap au marché aujourd'hui."
p = build_backtranslation_prompt(sample_resp)
assert sample_resp in p and len(p) > len(sample_resp) + 20

entry = create_sft_entry("Comment sont les prix au marché en ce moment ?", sample_resp)
assert "conversations" in entry and len(entry["conversations"]) == 2
print("🎉 TODO 2 validé !")

🎉 TODO 2 validé !


---
## 3. Déduplication & Split Train / Test sans fuite de données (`TODO 3`)

Complète le **`TODO 3`** pour dédupliquer les paires SFT, les mélanger aléatoirement, et les découper en `(sft_train, sft_test)`.

In [22]:
# === EXERCICE GUIDÉ : TODO 3 ===
def deduplicate_and_split_sft(
    sft_entries: List[Dict],
    test_ratio: float = 0.1,
    seed: int = 3407
) -> Tuple[List[Dict], List[Dict]]:
    """
    1. Supprime les doublons basés sur le texte de la réponse de l'assistant (en minuscules).
    2. Mélange la liste avec `random.Random(seed).shuffle(...)`.
    3. Découpe en `sft_train` (1 - test_ratio) et `sft_test` (test_ratio).
    """
    unique_entries = []
    seen_responses = set()

    # TODO 3.1 : Parcourir `sft_entries`, extraire la réponse de l'assistant `entry["conversations"][1]["content"].strip().lower()`.
    # Si elle n'est pas dans `seen_responses`, l'ajouter à `seen_responses` et garder `entry` dans `unique_entries`.
    
    # --- TON CODE ICI ---
    for entry in sft_entries: 
        assistant_response = entry["conversations"][1]["content"].strip().lower()
        if assistant_response not in seen_responses: 
            seen_responses.add(assistant_response)
            unique_entries.append(entry)
    # --------------------

    rng = random.Random(seed)
    rng.shuffle(unique_entries)

    # TODO 3.2 : Calculer `test_size = max(1, int(len(unique_entries) * test_ratio))` (si len >= 2),
    # et séparer `unique_entries` en `sft_train` et `sft_test`.
    sft_train, sft_test = [], []
    
    # --- TON CODE ICI ---
    if len(unique_entries)>= 2:
        test_size = max(1, int(len(unique_entries) * test_ratio))
        sft_test = unique_entries[:test_size]
        sft_train = unique_entries[test_size:]
    # -------------------- 

    return sft_train, sft_test

---
## 4. Fusion Locale avec `../conv_bot.json` & Envoi sur Hugging Face Hub (`TODO 4`)

Nous chargeons ton fichier local `../conv_bot.json`, créons les fichiers `../data/final/sft_train.jsonl`, `../data/final/sft_test.jsonl`, et publions sur Hugging Face Hub avec ton `HF_TOKEN` du fichier `../.env` !

In [25]:
# === EXERCICE GUIDÉ : TODO 4 ===
def build_hf_dataset_dict(
    cpt_texts: List[str],
    sft_train_list: List[Dict],
    sft_test_list: List[Dict]
) -> Dict[str, Dataset]:
    """
    Crée les Datasets Hugging Face à partir de nos listes Python :
      - 'cpt_train' : Dataset avec une colonne 'text'
      - 'sft_train' : Dataset avec une colonne 'conversations'
      - 'sft_test'  : Dataset avec une colonne 'conversations'
    """
    # TODO 4 : Utiliser `Dataset.from_list(...)` pour créer `cpt_train`, `sft_train` et `sft_test`.
    
    # --- TON CODE ICI ---
    cpt_train =  Dataset.from_list([{"text": text} for text in cpt_texts])
    sft_train = Dataset.from_list(sft_train_list)
    sft_test = Dataset.from_list(sft_test_list)
    return {
        "cpt_train": cpt_train,
        "sft_train": sft_train,
        "sft_test": sft_test
    }
    # --------------------

In [26]:
# 🧪 Assemblage réel avec ton ../conv_bot.json local et sauvegarde dans ../data/final/
conv_bot_path = PROJECT_ROOT / "conv_bot.json"
all_sft = []
if conv_bot_path.exists():
    with open(conv_bot_path, "r", encoding="utf-8") as f:
        txt = re.sub(r'//.*', '', f.read())
        txt = re.sub(r',\s*]', ']', txt)
        txt = re.sub(r',\s*}', '}', txt)
        all_sft.extend(json.loads(txt))

filtered_wa_path = PROJECT_ROOT / "data" / "processed" / "filtered_whatsapp_sft.json"
if filtered_wa_path.exists():
    with open(filtered_wa_path, "r", encoding="utf-8") as f:
        all_sft.extend(json.load(f))

sft_train, sft_test = deduplicate_and_split_sft(all_sft, test_ratio=0.1)
print(f"📊 Total SFT local dédupliqué : {len(sft_train)} train | {len(sft_test)} test")

# Sauvegarde locale de sft_test.jsonl pour l'évaluation locale à l'Étape 5
with open(FINAL_DATA_DIR / "sft_test.json", "w", encoding="utf-8") as f:
    json.dump(sft_test, f, indent=2, ensure_ascii=False)

# Pour pousser sur Hugging Face depuis ton PC (décommente une fois HF_TOKEN configuré dans ../.env) :
# hf_token = os.getenv("HF_TOKEN")
# ds_dict = build_hf_dataset_dict(["Exemple CPT"], sft_train, sft_test)
# DatasetDict({"train": ds_dict["sft_train"], "test": ds_dict["sft_test"]}).push_to_hub("Cmr237/camfranglais-corpus-sft", token=hf_token)
print("🎉 Étape 3 locale terminée ! Tu peux maintenant ouvrir le notebook de l'Étape 4 sur Google Colab (GPU).")

📊 Total SFT local dédupliqué : 180 train | 19 test
🎉 Étape 3 locale terminée ! Tu peux maintenant ouvrir le notebook de l'Étape 4 sur Google Colab (GPU).
